# Goal Spotting — R(2+1)D-18 on SoccerNet

Fine-tunes **R(2+1)D-18** (pretrained on Kinetics-400) to detect **goal events**.

## Pipeline
1. Load goal annotations from `manifest_goals.csv`
2. Build balanced clip dataset — jittered positives + hard negatives + random background
3. Fine-tune with class-weighted BCE loss + mixed precision
4. Slide a scoring window over each half → P(goal) over time
5. Smooth → peak-find → NMS → predicted timestamps
6. Grid-search post-processing params on validation
7. Export predictions to CSV

**Expected Drive layout**
```
<match_dir>/
  1_224p.mkv
  2_224p.mkv
  Labels-v2.json
```

**Tip:** Set `CFG.RESUME_CKPT` to skip training.

In [ ]:
# Run once per Colab session
!pip install -q scikit-learn opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, precision_recall_fscore_support


@dataclass
class CFG:
    # -- paths ---------------------------------------------------------------
    # Root folder containing all match subdirectories (each with Labels-v2.json
    # and 1_224p.mkv / 2_224p.mkv).  Set this and run the manifest builder cell
    # to auto-generate MANIFEST_PATH.
    DATA_ROOT: str = '/content/drive/MyDrive/aspotting/data'

    MANIFEST_PATH: str = '/content/drive/MyDrive/aspotting/manifests/manifest_goals.csv'
    # Labels-v2.json lives at <match_dir>/Labels-v2.json (match_dir from manifest).
    # Use DATA_ROOT_REMAP=(old_prefix, new_prefix) if you moved data.
    DATA_ROOT_REMAP: Optional[Tuple[str, str]] = None

    CKPT_DIR: str = '/content/drive/MyDrive/aspotting/checkpoints'
    OUT_DIR:  str = '/content/drive/MyDrive/aspotting/predictions'
    # Set to a .pt path to skip training and jump straight to inference
    RESUME_CKPT: Optional[str] = None

    # -- manifest building ---------------------------------------------------
    # Fraction of matches assigned to train / valid / test (must sum to 1.0)
    SPLIT_RATIOS: Tuple[float, float, float] = (0.7, 0.15, 0.15)
    # Video filename pattern inside each match folder (half number replaces {})
    VIDEO_FILENAME: str = '{}_224p.mkv'

    # -- clip extraction -----------------------------------------------------
    CLIP_SEC:   float = 8.0    # seconds of video per clip
    NUM_FRAMES: int   = 16     # frames sampled per clip
    IMG_SIZE:   int   = 112    # spatial crop (px)

    # -- training sample generation ------------------------------------------
    GOAL_JITTERS:      Tuple[float, ...] = (-2.0, 0.0, 2.0)
    NEG_HARD_PER_GOAL: int   = 3     # shots/corners per goal
    NEG_RAND_PER_GOAL: int   = 5     # random background clips per goal
    MIN_NEG_GAP_SEC:   float = 12.0  # negatives stay >= this far from goals

    # -- training ------------------------------------------------------------
    EPOCHS:       int   = 8
    BATCH_SIZE:   int   = 32
    LR:           float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    NUM_WORKERS:  int   = 8
    PRETRAINED:   bool  = True
    USE_SAMPLER:  bool  = True   # weighted sampler -> ~50/50 batches

    # -- sliding-window inference --------------------------------------------
    STRIDE_SEC:       float = 1.0
    INFER_BATCH_SIZE: int   = 128

    # -- post-processing grid ------------------------------------------------
    TOLERANCE_SEC:   float             = 5.0   # TP if pred within +-5 s of GT
    SMOOTH_WINDOWS:  Tuple[int, ...]   = (5, 7, 9)
    THRESHOLDS:      Tuple[float, ...] = field(
        default_factory=lambda: tuple(np.round(np.linspace(0.40, 0.90, 15), 3))
    )
    NMS_SEPARATIONS: Tuple[float, ...] = (55.0, 70.0, 90.0)
    TARGET_RECALL:   float             = 0.80

    # -- validation tuning speed ---------------------------------------------
    TUNE_MAX_HALVES:  Optional[int]   = 30    # None = all valid halves
    TUNE_STRIDE_SEC:  float           = 2.0   # coarser stride for sweep
    TUNE_MAX_SECONDS: Optional[float] = None  # None = full half

    SEED: int = 42


cfg = CFG()
os.makedirs(cfg.CKPT_DIR, exist_ok=True)
os.makedirs(cfg.OUT_DIR,  exist_ok=True)
os.makedirs(os.path.dirname(cfg.MANIFEST_PATH), exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device     :', device)
print('Data root  :', cfg.DATA_ROOT)
print('Manifest   :', cfg.MANIFEST_PATH)
print('Tolerance  : +-', cfg.TOLERANCE_SEC, 's')

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

## Helper functions

In [ ]:
# Events we treat as hard negatives (visually goal-like)
HARD_NEG_LABELS = {
    'shots on target', 'shots off target',
    'penalty', 'direct free-kick', 'corner',
}


def remap_path(p: str) -> str:
    if cfg.DATA_ROOT_REMAP is not None:
        old, new = cfg.DATA_ROOT_REMAP
        if p.startswith(old):
            return new + p[len(old):]
    return p


def parse_game_time(game_time: str):
    # '1 - 13:10' -> (half=1, seconds=790)
    if not isinstance(game_time, str) or '-' not in game_time:
        return None, None
    try:
        a, b = game_time.split('-', 1)
        half = int(a.strip())
        mm, ss = b.strip().split(':')
        return half, int(mm) * 60 + int(ss)
    except Exception:
        return None, None


def video_duration_sec(video_path: str) -> Optional[float]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    return float(n / fps) if n > 1 else None


def far_from_all(t: float, times: List[float], gap: float) -> bool:
    return all(abs(t - x) >= gap for x in times)


def sample_negatives(
    duration: float, n: int, forbidden: List[float],
    gap: float, clip_sec: float,
) -> List[float]:
    lo, hi = clip_sec / 2, duration - clip_sec / 2
    if hi <= lo or n <= 0:
        return []
    out, tried, forb = [], 0, list(forbidden)
    while len(out) < n and tried < 8000:
        tried += 1
        t = random.uniform(lo, hi)
        if far_from_all(t, forb, gap):
            out.append(t)
            forb.append(t)
    return out


def load_hard_negs_from_json(match_dir: str, half: int) -> List[float]:
    ann = os.path.join(match_dir, 'Labels-v2.json')
    if not os.path.exists(ann):
        return []
    try:
        with open(ann) as f:
            obj = json.load(f)
        times = []
        for ev in obj.get('annotations', []):
            label = str(ev.get('label', '')).strip().lower()
            h, t  = parse_game_time(ev.get('gameTime', ''))
            if h == half and t is not None and label in HARD_NEG_LABELS:
                times.append(float(t))
        return times
    except Exception:
        return []


print('Helpers ready.')

## Build manifest from DATA_ROOT

Scans `CFG.DATA_ROOT` recursively for match folders containing `Labels-v2.json`,
extracts all goal timestamps, assigns train/valid/test splits **by match** (not by
clip, to avoid data leakage), and saves to `CFG.MANIFEST_PATH`.

Skip this cell if you already have a manifest.

In [ ]:
def build_manifest(data_root: str, manifest_path: str) -> pd.DataFrame:
    """
    Walk data_root, find every Labels-v2.json, extract goal rows, assign splits
    by match (train/valid/test), save CSV.
    """
    GOAL_LABEL = 'goal'

    # 1. Discover all match dirs that have a Labels-v2.json
    match_dirs = []
    for dirpath, dirnames, filenames in os.walk(data_root):
        if 'Labels-v2.json' in filenames:
            match_dirs.append(dirpath)
    match_dirs.sort()
    print(f'Found {len(match_dirs)} match directories under {data_root}')
    if not match_dirs:
        raise RuntimeError(
            f'No Labels-v2.json files found under {data_root}. '
            'Check that DATA_ROOT points to the right folder.')

    # 2. Assign splits by match (shuffle then slice)
    rng = random.Random(cfg.SEED)
    shuffled = list(match_dirs)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(round(n * cfg.SPLIT_RATIOS[0])))
    n_valid = max(1, int(round(n * cfg.SPLIT_RATIOS[1])))
    split_map = {}
    for d in shuffled[:n_train]:
        split_map[d] = 'train'
    for d in shuffled[n_train:n_train + n_valid]:
        split_map[d] = 'valid'
    for d in shuffled[n_train + n_valid:]:
        split_map[d] = 'test'

    # 3. Parse each Labels-v2.json and emit one row per (match, half, goal)
    rows = []
    skipped_matches = 0
    for match_dir in tqdm(match_dirs, desc='Parsing annotations'):
        split = split_map[match_dir]
        ann_path = os.path.join(match_dir, 'Labels-v2.json')
        try:
            with open(ann_path) as f:
                obj = json.load(f)
        except Exception as e:
            print(f'  [skip] cannot read {ann_path}: {e}')
            skipped_matches += 1
            continue

        for ev in obj.get('annotations', []):
            if str(ev.get('label', '')).strip().lower() != GOAL_LABEL:
                continue
            half, sec = parse_game_time(ev.get('gameTime', ''))
            if half is None or sec is None:
                continue

            video_path = os.path.join(
                match_dir, cfg.VIDEO_FILENAME.format(half))
            if not os.path.exists(video_path):
                # Try common alternate names
                for alt in [f'{half}_720p.mkv', f'{half}.mkv',
                            f'{half}_224p.mp4', f'{half}.mp4']:
                    alt_path = os.path.join(match_dir, alt)
                    if os.path.exists(alt_path):
                        video_path = alt_path
                        break
                else:
                    # File genuinely missing — still record it so the manifest
                    # is complete; load_halves_from_manifest will skip it.
                    pass

            rows.append({
                'split':      split,
                'match_dir':  match_dir,
                'video_path': video_path,
                'half':       half,
                'center_sec': float(sec),
                'label':      1,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            'No goal annotations found. '
            'Check that Labels-v2.json files use label="goal".')

    # Summary
    for s in ('train', 'valid', 'test'):
        sub = df[df.split == s]
        n_matches = sub['match_dir'].nunique()
        print(f'  {s:6s}: {n_matches:3d} matches, {len(sub):4d} goals')
    if skipped_matches:
        print(f'  WARNING: {skipped_matches} match(es) skipped due to read errors')

    df.to_csv(manifest_path, index=False)
    print(f'\nManifest saved -> {manifest_path}')
    return df


# Comment out this call if you already have a manifest and don't want to rebuild.
manifest_df = build_manifest(cfg.DATA_ROOT, cfg.MANIFEST_PATH)

## Load annotations from manifest

In [ ]:
def load_halves_from_manifest(manifest_path: str) -> Dict[str, List[dict]]:
    # Reads manifest_goals.csv columns: split, match_dir, video_path, half, center_sec, label
    # Groups rows into half-info dicts keyed by split.
    df = pd.read_csv(manifest_path)
    df['match_dir']  = df['match_dir'].apply(remap_path)
    df['video_path'] = df['video_path'].apply(remap_path)

    splits: Dict[str, List[dict]] = {}
    groups = df.groupby(['split', 'match_dir', 'video_path', 'half'])

    for (split, match_dir, video_path, half), grp in tqdm(groups, desc='Loading halves'):
        half = int(half)
        dur  = video_duration_sec(video_path)
        if dur is None:
            print(f'  [skip] cannot open: {video_path}')
            continue
        splits.setdefault(split, []).append({
            'split':        split,
            'match_dir':    match_dir,
            'video_path':   video_path,
            'half':         half,
            'duration_sec': dur,
            'goal_times':   sorted(grp['center_sec'].tolist()),
            'hard_times':   sorted(load_hard_negs_from_json(match_dir, half)),
        })

    for s, v in splits.items():
        n_g = sum(len(h['goal_times']) for h in v)
        print(f'  {s:6s}: {len(v):3d} halves, {n_g:4d} goals')
    return splits


all_halves   = load_halves_from_manifest(cfg.MANIFEST_PATH)
train_halves = all_halves.get('train', [])
valid_halves = all_halves.get('valid', [])
test_halves  = all_halves.get('test',  [])
print(f'train={len(train_halves)}  valid={len(valid_halves)}  test={len(test_halves)}')

## Build clip-level dataset

In [ ]:
def build_clip_df(halves: List[dict], split_name: str) -> pd.DataFrame:
    rows    = []
    jitters = cfg.GOAL_JITTERS if split_name == 'train' else (0.0,)
    half_s  = cfg.CLIP_SEC / 2

    for h in tqdm(halves, desc=f'Building {split_name} clips'):
        dur   = h['duration_sec']
        goals = h['goal_times']
        hard  = h['hard_times']

        # positives: goal centre +- jitter
        for g in goals:
            for j in jitters:
                c = float(np.clip(g + j, half_s, dur - half_s))
                rows.append({'video_path': h['video_path'],
                             'center_sec': c, 'label': 1})

        # hard negatives: shots/corners far from goals
        hard_ok = [t for t in hard
                   if far_from_all(t, goals, cfg.MIN_NEG_GAP_SEC)]
        n_hard  = min(len(hard_ok), cfg.NEG_HARD_PER_GOAL * max(1, len(goals)))
        for t in (random.sample(hard_ok, n_hard) if n_hard else []):
            c = float(np.clip(t, half_s, dur - half_s))
            rows.append({'video_path': h['video_path'],
                         'center_sec': c, 'label': 0})

        # random background negatives
        n_rand = cfg.NEG_RAND_PER_GOAL * max(1, len(goals))
        for t in sample_negatives(
                dur, n_rand, goals + hard_ok,
                cfg.MIN_NEG_GAP_SEC, cfg.CLIP_SEC):
            rows.append({'video_path': h['video_path'],
                         'center_sec': t, 'label': 0})

    df = pd.DataFrame(rows).drop_duplicates()
    pos = int(df.label.sum())
    print(f'  {split_name}: {len(df)} clips  (pos={pos}, neg={len(df)-pos})')
    return df


train_df = build_clip_df(train_halves, 'train')
valid_df = build_clip_df(valid_halves, 'valid')

train_df.to_csv(os.path.join(cfg.OUT_DIR, 'clips_train.csv'), index=False)
valid_df.to_csv(os.path.join(cfg.OUT_DIR, 'clips_valid.csv'), index=False)

## Pre-extract clips to local disk  *(run once, then training is much faster)*

Reads every clip from Drive video files and saves it as a `.pt` tensor on local
disk.  Subsequent runs skip files that already exist.  Training then loads tensors
directly instead of seeking into videos — typically 10-20× faster IO.

In [ ]:
CLIP_CACHE_DIR = '/content/clip_cache'   # local disk — fast, but lost on runtime restart
os.makedirs(CLIP_CACHE_DIR, exist_ok=True)


def clip_cache_path(video_path: str, center_sec: float) -> str:
    # Stable filename: hash of (video_path, center_sec)
    import hashlib
    key = f'{video_path}@{center_sec:.3f}'
    h   = hashlib.md5(key.encode()).hexdigest()
    return os.path.join(CLIP_CACHE_DIR, f'{h}.pt')


def pre_extract_clips(df: pd.DataFrame, desc: str = 'Pre-extracting') -> int:
    """Extract every clip in df and cache to local disk. Returns number extracted."""
    extracted = skipped = errors = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc=desc):
        p = clip_cache_path(r.video_path, r.center_sec)
        if os.path.exists(p):
            skipped += 1
            continue
        try:
            t = extract_clip(r.video_path, r.center_sec)
            torch.save(t, p)
            extracted += 1
        except Exception as e:
            errors += 1
    print(f'  extracted={extracted}  already_cached={skipped}  errors={errors}')
    return extracted


print('Pre-extracting train clips...')
pre_extract_clips(train_df, 'Train clips')
print('Pre-extracting valid clips...')
pre_extract_clips(valid_df, 'Valid clips')

## Dataset and DataLoaders

In [ ]:
def extract_clip(video_path: str, center_sec: float) -> torch.Tensor:
    # Sample NUM_FRAMES around center_sec -> (C, T, H, W)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open: {video_path}')
    fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    start = max(0.0, center_sec - cfg.CLIP_SEC / 2)
    step  = max(1, int(cfg.CLIP_SEC * fps / cfg.NUM_FRAMES))
    frames = []
    for i in range(cfg.NUM_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start * fps) + i * step)
        ok, fr = cap.read()
        if not ok:
            break
        fr = cv2.resize(fr, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
        fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        frames.append(fr)
    cap.release()
    if not frames:
        raise RuntimeError(f'No frames read: {video_path}')
    while len(frames) < cfg.NUM_FRAMES:
        frames.append(frames[-1].copy())
    arr = np.stack(frames).astype(np.float32) / 255.0
    return torch.from_numpy(np.transpose(arr, (3, 0, 1, 2)))  # (C,T,H,W)


class GoalDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        try:
            p = clip_cache_path(r.video_path, r.center_sec)
            if os.path.exists(p):
                clip = torch.load(p, weights_only=True)
            else:
                clip = extract_clip(r.video_path, r.center_sec)
        except Exception as e:
            print(f'  [skip] {e}')
            return None
        return (clip, torch.tensor(float(r.label), dtype=torch.float32))


def collate_skip_none(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return torch.utils.data.dataloader.default_collate(batch)


def make_loader(df, shuffle=False, sampler=None):
    return DataLoader(
        GoalDataset(df),
        batch_size=cfg.BATCH_SIZE,
        shuffle=(shuffle and sampler is None),
        sampler=sampler,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=cfg.NUM_WORKERS > 0,
        collate_fn=collate_skip_none,
    )


if cfg.USE_SAMPLER:
    labels  = train_df['label'].values.astype(int)
    counts  = np.bincount(labels, minlength=2)
    weights = np.where(labels == 1,
                       1.0 / max(counts[1], 1),
                       1.0 / max(counts[0], 1))
    sampler      = WeightedRandomSampler(
        torch.from_numpy(weights).double(), len(weights), replacement=True)
    train_loader = make_loader(train_df, sampler=sampler)
else:
    train_loader = make_loader(train_df, shuffle=True)

valid_loader = make_loader(valid_df)
print(f'Train batches: {len(train_loader)},  Valid batches: {len(valid_loader)}')

## Model + Training

Skipped automatically when `CFG.RESUME_CKPT` is set.

In [ ]:
best_ckpt   = os.path.join(cfg.CKPT_DIR, 'goalspotter_best.pt')
latest_ckpt = os.path.join(cfg.CKPT_DIR, 'goalspotter_latest.pt')


def build_model() -> nn.Module:
    w = R2Plus1D_18_Weights.DEFAULT if cfg.PRETRAINED else None
    m = r2plus1d_18(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m.to(device)


def eval_clip_metrics(model, loader, thr=0.5):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            x, y = batch
            logits = model(x.to(device)).squeeze(1)
            ps.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(y.numpy())
    y_true = np.concatenate(ys)
    y_prob = np.concatenate(ps)
    y_hat  = (y_prob >= thr).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true.astype(int), y_hat, average='binary', zero_division=0)
    ap = average_precision_score(y_true.astype(int), y_prob)
    return {'auprc': float(ap), 'precision': float(prec),
            'recall': float(rec), 'f1': float(f1)}


if cfg.RESUME_CKPT and os.path.exists(cfg.RESUME_CKPT):
    print('RESUME_CKPT found — skipping training.')
else:
    model = build_model()

    n_pos = float(train_df.label.sum())
    n_neg = float((train_df.label == 0).sum())
    pos_w = torch.tensor([n_neg / max(n_pos, 1.0)], device=device)
    print(f'pos_weight = {float(pos_w):.2f}  (neg={n_neg:.0f} / pos={n_pos:.0f})')
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    best_auprc = -1.0
    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        total_loss = 0.0
        step = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.EPOCHS}')
        for batch in pbar:
            if batch is None:
                continue
            x, y = batch
            step += 1
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                loss = criterion(model(x).squeeze(1), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/step:.4f}')

        val_m = eval_clip_metrics(model, valid_loader)
        print(f'Epoch {epoch}  AUPRC={val_m["auprc"]:.4f}  '
              f'P={val_m["precision"]:.3f}  '
              f'R={val_m["recall"]:.3f}  '
              f'F1={val_m["f1"]:.3f}')

        ckpt = {'epoch': epoch, 'model': model.state_dict(), 'cfg': cfg.__dict__}
        torch.save(ckpt, latest_ckpt)
        if val_m['auprc'] > best_auprc:
            best_auprc = val_m['auprc']
            torch.save(ckpt, best_ckpt)
            print(f'  -> saved best (AUPRC {best_auprc:.4f})')

    print('Training complete.')

In [ ]:
load_path = (cfg.RESUME_CKPT
             if (cfg.RESUME_CKPT and os.path.exists(cfg.RESUME_CKPT))
             else best_ckpt)

model = build_model()
ckpt  = torch.load(load_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded: {load_path}  (epoch {ckpt.get("epoch", "?")})')

## Sliding-window inference + post-processing

In [ ]:
@torch.no_grad()
def score_half(
    video_path: str,
    stride_sec: float = None,
    max_sec:    float = None,
) -> Tuple[np.ndarray, np.ndarray]:
    # Slide a clip window; returns (centers, probs)
    stride_sec = stride_sec or cfg.STRIDE_SEC
    dur = video_duration_sec(video_path)
    if dur is None:
        return np.array([]), np.array([])
    if max_sec:
        dur = min(dur, float(max_sec))
    t0 = cfg.CLIP_SEC / 2
    t1 = max(t0, dur - cfg.CLIP_SEC / 2)
    centers = np.arange(t0, t1 + 1e-6, stride_sec, dtype=np.float32)
    if len(centers) == 0:
        return np.array([]), np.array([])

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return np.array([]), np.array([])
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    probs, batch = [], []

    def _flush():
        if not batch:
            return
        x = torch.stack(batch).to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            p = torch.sigmoid(model(x).squeeze(1)).cpu().numpy().tolist()
        probs.extend(p)
        batch.clear()

    for c in centers:
        start  = max(0.0, float(c) - cfg.CLIP_SEC / 2)
        step   = max(1, int(cfg.CLIP_SEC * fps / cfg.NUM_FRAMES))
        frames = []
        for i in range(cfg.NUM_FRAMES):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(start * fps) + i * step)
            ok, fr = cap.read()
            if not ok:
                break
            fr = cv2.resize(fr, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            frames.append(fr)
        if not frames:
            probs.append(0.0)
            continue
        while len(frames) < cfg.NUM_FRAMES:
            frames.append(frames[-1].copy())
        arr = np.stack(frames).astype(np.float32) / 255.0
        batch.append(torch.from_numpy(np.transpose(arr, (3, 0, 1, 2))))
        if len(batch) >= cfg.INFER_BATCH_SIZE:
            _flush()

    _flush()
    cap.release()
    m = min(len(probs), len(centers))
    return centers[:m], np.array(probs[:m], dtype=np.float32)


def moving_avg(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1 or len(x) == 0:
        return x.copy()
    return np.convolve(
        np.pad(x, (k // 2, k // 2), 'edge'),
        np.ones(k, np.float32) / k, 'valid')


def nms_1d(
    times: np.ndarray, scores: np.ndarray, min_sep: float
) -> Tuple[np.ndarray, np.ndarray]:
    if len(times) == 0:
        return np.array([]), np.array([])
    order      = np.argsort(scores)[::-1]
    kept       = []
    suppressed = np.zeros(len(times), bool)
    for i in order:
        if suppressed[i]:
            continue
        kept.append(i)
        suppressed |= np.abs(times - times[i]) < min_sep
        suppressed[i] = False
    kept = np.array(kept)
    s    = np.argsort(times[kept])
    return times[kept][s], scores[kept][s]


def detect_events(
    centers: np.ndarray, probs: np.ndarray,
    threshold: float, smooth_k: int, nms_sep: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Returns (smoothed_probs, pred_times, pred_scores)
    if len(centers) == 0:
        return np.array([]), np.array([]), np.array([])
    sm  = moving_avg(probs, smooth_k)
    idx = [
        i for i in range(len(sm))
        if sm[i] >= threshold
        and sm[i] >= (sm[i - 1] if i > 0           else -1)
        and sm[i] >= (sm[i + 1] if i < len(sm) - 1 else -1)
    ]
    if not idx:
        return sm, np.array([]), np.array([])
    t, s = nms_1d(centers[idx], sm[np.array(idx)], min_sep=nms_sep)
    return sm, t, s


def match_events(
    pred_t, pred_s, gt_t, tol: float
) -> Tuple[int, int, int]:
    # Greedy score-ordered matching. Returns (TP, FP, FN).
    if not len(pred_t):
        return 0, 0, len(gt_t)
    pred_t = np.asarray(pred_t, np.float32)
    pred_s = np.asarray(pred_s, np.float32)
    gt_t   = np.asarray(gt_t,   np.float32)
    used   = np.zeros(len(gt_t), bool)
    tp = fp = 0
    for i in np.argsort(pred_s)[::-1]:
        if len(gt_t) == 0:
            fp += 1
            continue
        d = np.abs(gt_t - pred_t[i])
        j = int(np.argmin(d))
        if d[j] <= tol and not used[j]:
            tp += 1
            used[j] = True
        else:
            fp += 1
    return tp, fp, int((~used).sum())


print('Inference helpers ready.')

## Parameter sweep on validation

Score a subset of halves once, then sweep threshold/smooth/NMS cheaply.

In [ ]:
pool = [h for h in valid_halves if h['goal_times']] or list(valid_halves)
if cfg.TUNE_MAX_HALVES and cfg.TUNE_MAX_HALVES < len(pool):
    rng  = random.Random(cfg.SEED)
    pool = rng.sample(pool, cfg.TUNE_MAX_HALVES)

print(f'Pre-scoring {len(pool)} / {len(valid_halves)} valid halves '
      f'(stride={cfg.TUNE_STRIDE_SEC}s) ...')

model.eval()
streams = []
for h in tqdm(pool):
    c, p = score_half(h['video_path'],
                      stride_sec=cfg.TUNE_STRIDE_SEC,
                      max_sec=cfg.TUNE_MAX_SECONDS)
    gt   = [t for t in h['goal_times']
            if cfg.TUNE_MAX_SECONDS is None or t <= cfg.TUNE_MAX_SECONDS]
    streams.append({'goal_times': gt, 'centers': c, 'probs': p,
                    'match_dir': h['match_dir'], 'half': h['half']})

n_gt = sum(len(s['goal_times']) for s in streams)
print(f'Done.  {n_gt} GT goals across {len(streams)} halves.')

In [ ]:
def evaluate_params(streams, thr, k, nms_sep):
    tp = fp = fn = 0
    for s in streams:
        _, pt, ps = detect_events(s['centers'], s['probs'], thr, k, nms_sep)
        a, b, c   = match_events(pt, ps, s['goal_times'], cfg.TOLERANCE_SEC)
        tp += a; fp += b; fn += c
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-8)
    return {'threshold': thr, 'smooth_k': k, 'nms_sep': nms_sep,
            'precision': prec, 'recall': rec, 'f1': f1,
            'fp_per_half': fp / max(len(streams), 1),
            'tp': tp, 'fp': fp, 'fn': fn}


rows  = [evaluate_params(streams, thr, k, nms)
         for thr in cfg.THRESHOLDS
         for k   in cfg.SMOOTH_WINDOWS
         for nms in cfg.NMS_SEPARATIONS]

sweep = pd.DataFrame(rows).sort_values(
    ['recall', 'fp_per_half', 'f1'], ascending=[False, True, False])

ok = sweep[sweep['recall'] >= cfg.TARGET_RECALL]
best_row = (ok.sort_values(['fp_per_half', 'f1'], ascending=[True, False]).iloc[0]
            if len(ok) else sweep.iloc[0])

cols = ['threshold', 'smooth_k', 'nms_sep', 'precision', 'recall', 'f1', 'fp_per_half']
print(f'Searched {len(rows)} configs.  Top 10:')
display(sweep[cols].head(10))
print('\nSelected:')
print(best_row[cols].to_dict())

best_params = {
    'threshold': float(best_row['threshold']),
    'smooth_k':  int(best_row['smooth_k']),
    'nms_sep':   float(best_row['nms_sep']),
}

## Visualise a validation half

In [ ]:
import matplotlib.pyplot as plt

IDX = 0  # change to inspect a different stream
s   = streams[IDX]

sm, pred_t, pred_s = detect_events(s['centers'], s['probs'], **best_params)

print(f'Match   : {s["match_dir"]}')
print(f'Half    : {s["half"]}')
print(f'GT  (s) : {[round(float(x), 1) for x in s["goal_times"]]}')
print(f'Pred (s): {[round(float(x), 1) for x in pred_t.tolist()]}')

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(s['centers'], s['probs'], alpha=0.3,  lw=1,   label='raw P(goal)')
ax.plot(s['centers'], sm,         alpha=0.9,  lw=1.5, label=f'smooth k={best_params["smooth_k"]}')
if len(pred_t):
    ax.scatter(pred_t, pred_s, marker='x', s=120, color='red',
               zorder=5, label='prediction')
for gt in s['goal_times']:
    ax.axvline(gt, color='green', lw=1.2, linestyle='--', alpha=0.7)
ax.axhline(best_params['threshold'], color='gray', linestyle=':',
           label=f'threshold={best_params["threshold"]}')
ax.set(xlim=(s['centers'][0], s['centers'][-1]), ylim=(0, 1),
       xlabel='Time (seconds)', ylabel='P(goal)',
       title=f'Goal probability stream — half {s["half"]}')
ax.legend()
plt.tight_layout()
plt.show()

## Full validation evaluation

*(Uses `cfg.STRIDE_SEC`, not the faster tuning stride.)*

In [ ]:
model.eval()
tp = fp = fn = 0

for h in tqdm(valid_halves, desc='Evaluating valid'):
    c, p      = score_half(h['video_path'])
    _, pt, ps = detect_events(c, p, **best_params)
    a, b, cc  = match_events(pt, ps, h['goal_times'], cfg.TOLERANCE_SEC)
    tp += a; fp += b; fn += cc

prec = tp / max(tp + fp, 1)
rec  = tp / max(tp + fn, 1)
f1   = 2 * prec * rec / max(prec + rec, 1e-8)
fph  = fp / max(len(valid_halves), 1)

print(f'Validation event metrics  (tolerance +/-{cfg.TOLERANCE_SEC}s)')
print(f'  Precision  : {prec:.3f}')
print(f'  Recall     : {rec:.3f}')
print(f'  F1         : {f1:.3f}')
print(f'  FP / half  : {fph:.2f}')
print(f'  TP/FP/FN   : {tp}/{fp}/{fn}')

## Results plots — Confusion matrix, PR curve, ROC curve

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Aggregate sweep: best F1 per threshold (across smooth_k / nms_sep combos)
sweep_agg = (sweep.groupby('threshold', as_index=False)
             .apply(lambda g: g.loc[g['f1'].idxmax()])
             .sort_values('threshold')
             .reset_index(drop=True))

# Best recall threshold (from best_params) vs best F1 threshold
recall_row = sweep_agg.loc[sweep_agg['threshold'] == best_params['threshold']].iloc[0]
f1_row     = sweep_agg.loc[sweep_agg['f1'].idxmax()]

fig, axes = plt.subplots(1, 5, figsize=(28, 5))

# --- Event-level PR curve ---
axes[0].plot(sweep_agg['recall'], sweep_agg['precision'], 'o-', ms=4)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Event-level Precision–Recall')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# --- F1 vs threshold ---
axes[1].plot(sweep_agg['threshold'], sweep_agg['f1'], 'o-', ms=4)
axes[1].axvline(best_params['threshold'], color='red',    linestyle='--', label=f'recall threshold={best_params["threshold"]}')
axes[1].axvline(f1_row['threshold'],      color='orange', linestyle='--', label=f'F1 threshold={f1_row["threshold"]}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1')
axes[1].set_title('F1 vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Event-level CM at best recall threshold ---
def plot_event_cm(ax, row, title):
    tp, fp, fn = int(row['tp']), int(row['fp']), int(row['fn'])
    cm = np.array([[tp, fn], [fp, 0]])
    ConfusionMatrixDisplay(cm, display_labels=['Goal', 'Background']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plot_event_cm(axes[2], recall_row,
              f'Event-level (recall thr={best_params["threshold"]})\n'
              f'P={recall_row["precision"]:.2f} R={recall_row["recall"]:.2f} F1={recall_row["f1"]:.2f}')

plot_event_cm(axes[3], f1_row,
              f'Event-level (F1 thr={f1_row["threshold"]})\n'
              f'P={f1_row["precision"]:.2f} R={f1_row["recall"]:.2f} F1={f1_row["f1"]:.2f}')

# --- Clip-level confusion matrix ---
CLIP_THRESHOLD = 0.9  # change this to experiment

model.eval()
all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(valid_loader, desc='Clip scores'):
        if batch is None:
            continue
        x, y = batch
        logits = model(x.to(device)).squeeze(1)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

y_true = np.array(all_labels)
y_pred = (np.array(all_probs) >= CLIP_THRESHOLD).astype(int)
clip_cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(clip_cm, display_labels=['Background', 'Goal']).plot(
    ax=axes[4], colorbar=False, cmap='Blues')
axes[4].set_title(f'Clip-level (threshold={CLIP_THRESHOLD})')

plt.tight_layout()
plt.show()

## Export predicted timestamps

In [ ]:
def sec_to_mmss(sec: float) -> str:
    s = max(0, int(round(sec)))
    return f'{s // 60:02d}:{s % 60:02d}'


def export_predictions(halves: List[dict], split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    for h in tqdm(halves, desc=f'Exporting {split_name}'):
        c, p      = score_half(h['video_path'])
        _, pt, ps = detect_events(c, p, **best_params)
        for t, sc in zip(pt.tolist(), ps.tolist()):
            rows.append({
                'split':      split_name,
                'match_dir':  h['match_dir'],
                'half':       h['half'],
                'pred_sec':   round(float(t), 2),
                'pred_mmss':  sec_to_mmss(float(t)),
                'confidence': round(float(sc), 4),
            })
    out = os.path.join(cfg.OUT_DIR, f'predicted_goals_{split_name}.csv')
    df  = pd.DataFrame(rows)
    df.to_csv(out, index=False)
    print(f'Saved {len(df)} predictions -> {out}')
    return df


pred_valid = export_predictions(valid_halves, 'valid')
pred_test  = export_predictions(test_halves,  'test')
display(pred_valid.head(20))

## Quick tuning reference

| Goal | Change in `CFG` |
|------|-----------------|
| Higher recall | Lower `THRESHOLDS` min, reduce `MIN_NEG_GAP_SEC` |
| Fewer false alarms | Increase `NMS_SEPARATIONS`, add more `NEG_HARD_PER_GOAL` |
| Faster inference | Raise `STRIDE_SEC` to 1.5–2.0 |
| Better temporal precision | Lower `STRIDE_SEC`, raise `NUM_FRAMES` |
| Resume after crash | `RESUME_CKPT = '.../goalspotter_latest.pt'` |
| Re-use saved model | `RESUME_CKPT = '.../goalspotter_best.pt'` |